# Causal discovery

교육 분야의 중요한 과제 중 하나는 개인별 최적 학습 순서를 설계하는 것입니다.

이를 위해 실제 온라인 교육 플랫폼 Eedi의 로그를 기반으로 구축된 CausalEdu 데이터셋을 사용하여,

Discovery → Identification → Estimation → Refutation의 전 과정을 PyWhy 스택으로 구현합니다.

In [3]:
%pip -q install lingam

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os, warnings, numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

# Visualization
import matplotlib.pyplot as plt
import networkx as nx

# PyWhy Core Stack
from dowhy import CausalModel
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
import lingam

# Random Seed & Config
SEED = 18
np.random.seed(SEED)
plt.style.use("default")
DATA_DIR = Path("../data/causal_edu")

## Data Setup

CausalEdu는 관찰 데이터 및 일부 학습 개념 쌍(construct pairs)에 대해 A/B 테스트와 전문가 지식을 포함한 데이터셋입니다.

### checkins_lessons_checkouts_training.csv - 관찰 로그(Training)

Eedi 플랫폼에서 수집된 학생별 학습 로그입니다.  
각 세션은 Check-in → Lesson → Check-out의 순서로 구성됩니다.

- 각 행(row)은 학생의 한 번의 응답 이벤트
- `Type` 열은 이벤트 유형을 구분 (Checkin / CheckinRetry / Lesson / Checkout / CheckoutRetry)  
- `ConstructId`는 학습 개념(예: ‘분수 덧셈’)을 나타냄  

이후 인과 구조 학습 단계에 활용됩니다.

In [5]:
training = pd.read_csv(DATA_DIR / "checkins_lessons_checkouts_training.csv", parse_dates=["Timestamp"])
print("training:", training.shape)
print("range:", training["Timestamp"].min(), "~", training["Timestamp"].max())
display(training.head(3))

training: (641490, 12)
range: 2022-02-01 01:53:31.170000 ~ 2022-09-30 11:03:43.307000


,QuizSessionId,AnswerId,UserId,QuizId,QuestionId,IsCorrect,AnswerValue,CorrectAnswer,QuestionSequence,ConstructId,Type,Timestamp
0,0,0.0,0,242762,130326,1.0,4.0,4.0,1,9,Checkin,2022-02-01 01:53:31.170
1,1,1.0,1,242762,130326,1.0,4.0,4.0,1,9,Checkin,2022-02-01 02:07:31.393
2,1,2.0,1,242762,130327,0.0,1.0,4.0,2,10,Checkin,2022-02-01 02:08:27.947


### checkin_to_checkout.csv - 인과발견(Discovery)용 Ground Truth

특정 LessonConstructId → QuestionConstructId 페어에 대해 `n00, n01, n10, n11`(예 n00: 오답, 오답)을 제공하고, 

두 가지 가설로부터 **p010_m**, **k010_m** 통계를 제공합니다.
- 가설 1(학습은 해롭지 않다): $p010\_m = \frac{n_{01}}{n_{00}+n_{01}}$
- 가설 2(우연 정답 보정): $k010\_m = \frac{n_{01}}{n_{00}+n_{01}+n_{10}}$

임계치 **0.25**(4지선다 랜덤 추측 배제) 초과 시 **선행관계(Prerequisite)**로 간주합니다.


In [11]:
c2c = pd.read_csv(DATA_DIR / "checkin_to_checkout.csv")
print("c2c:", c2c.shape)
display(c2c.head(3))

c2c: (183, 9)


,LessonConstructId,QuestionConstructId,n00,n01,n10,n11,Count,p010_m,k010_m
0,70,1270,8,38,3,30,79,0.826087,0.775510
1,70,1271,56,5,6,11,78,0.081967,0.074627
2,70,1272,13,10,18,36,77,0.434783,0.243902


### construct_experiments_ates_test.csv - 인과추론(Inference)용 Ground Truth (CATE)

A/B 실험을 통해 측정된 **조건부 평균처치효과(CATE)** 데이터입니다.  
특정 레슨 개념(`TreatmentLessonConstructId`)이  
다른 개념(`QuestionConstructId`)의 성취도에 미치는 실제 개입 효과를 `ate_p_1_`, `ate_k_1_`로 제공합니다.  

DoWhy로 추정한 인과효과와 비교해 모델의 정확도를 검증하는 데 사용됩니다.

In [8]:
ab_ates = pd.read_csv(DATA_DIR / "construct_experiments_ates_test.csv")
print("ab_ates:", ab_ates.shape)
display(ab_ates.head(3))


ab_ates: (88, 8)


,TreatmentLessonConstructId,QuestionConstructId,Year,ControlLessonConstructIds,ControlUsersCount,TreatmentUsersCount,ate_p_1__,ate_k_1__
0,206,211,7,{3119},73,94,0.033656,-0.019091
1,206,212,7,{3119},77,101,-0.022222,-0.036004
2,206,216,7,{3119},75,94,-0.109501,-0.118014


### construct_prerequisites_test.csv - 전문가 지식 그래프(Prior knowledge)

전문가가 설계한 선행 학습 개념 그래프입니다.  
각 `ConstructId`는 해당 과목(`SubjectId`) 내에서  
`PrerequisiteConstructIds`(부모 노드 집합)과 연결됩니다.

인과 그래프 학습 시 다음과 같이 활용됩니다:  
- `require_edges`: 전문가 그래프에 존재하는 엣지는 반드시 포함  
- `forbid_edges`: 상반되는 방향의 엣지는 금지  
- 전문가 그래프와 A/B 결과가 충돌할 경우 **A/B 실험 결과를 우선**으로 신뢰ㅋ

In [9]:
expert = pd.read_csv(DATA_DIR / "construct_prerequisites_test.csv")
print("expert:", expert.shape)
display(expert.head(3))

expert: (3277, 3)


,ConstructId,SubjectId,PrerequisiteConstructIds
0,854,33.0,{76}
1,855,33.0,{76}
2,856,33.0,"{483, 76}"
